In [ ]:
import eos
import os
import numpy as np
import matplotlib.pyplot as plt
from wquantiles import quantile_1D
import ast
import cmath
import re
import pandas as pd
from scipy.stats import chi2

colors = eos.figure.ItemColorCycler()._colors

resample = True

In [ ]:
BASE_DIRECTORY = os.getcwd()
if not os.path.exists(BASE_DIRECTORY):
    os.makedirs(os.path.abspath(BASE_DIRECTORY), exist_ok=True)

In [ ]:
def corner_plot(path):
    figure_args = {
        'type': 'corner',
        'contents': [{
            'path': path,
            'label': 'Global fit'
        }]
    }
    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()

In [ ]:
def wstd(samples, weights=None, axis=0):
    mean_w = np.average(samples, weights=weights, axis=axis)
    return np.sqrt(np.average((samples - mean_w)**2, weights=weights, axis=axis))

def wmedian(samples, weights=None):
    samples = np.asarray(samples)

    if weights is None:
        return np.median(samples)

    weights = np.asarray(weights)
    if np.any(weights < 0) or not np.any(weights > 0):
        raise ValueError("Weights must be nonnegative with a positive total.")

    keep = weights > 0
    samples, weights = samples[keep], weights[keep]

    order = np.argsort(samples)
    samples, weights = samples[order], weights[order]

    positions = (np.cumsum(weights) - 0.5 * weights) / np.sum(weights)
    return np.interp(0.5, positions, samples)


def wquantiles(samples, weights=None, quantiles=(0.16, 0.50, 0.84)):
    samples = np.asarray(samples)

    if weights is None:
        return np.quantile(samples, quantiles)

    weights = np.asarray(weights)
    if np.any(weights < 0) or not np.any(weights > 0):
        raise ValueError("Weights must be nonnegative with a positive total.")

    keep = weights > 0
    samples, weights = samples[keep], weights[keep]

    order = np.argsort(samples)
    samples, weights = samples[order], weights[order]

    positions = (np.cumsum(weights) - 0.5 * weights) / np.sum(weights)
    return np.interp(quantiles, positions, samples)


def get_gof(analysis, param_array):
    parameter_list = []

    for param in analysis.init_args['priors']:
        if param['type'] == 'transform':
            parameter_list += param['parameters']
        else:
            parameter_list.append(param['parameter'])

    for i, param in enumerate(parameter_list):
        analysis.parameters.set(param, param_array[i])

    return analysis.goodness_of_fit()

In [ ]:
def get_f_rho(sample, ana):
    get_gof(ana, sample)
    s_rho = (sample[0] - 1j * sample[1] / 2)**2
    return [eos.Observable.make("0->pipi::Re{f_+}(Re{q2},Im{q2})",
                        ana.parameters,
                        eos.Kinematics({"Re{q2}": np.real(s_rho), "Im{q2}": np.imag(s_rho)}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate(),
            eos.Observable.make("0->pipi::Im{f_+}(Re{q2},Im{q2})",
                        ana.parameters,
                        eos.Kinematics({"Re{q2}": np.real(s_rho), "Im{q2}": np.imag(s_rho)}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()]

def get_f_bfp(ana, bfp, q2):
    get_gof(ana, bfp)
    return eos.Observable.make("0->pipi::Abs{f_+}^2(q2)",
                        ana.parameters,
                        eos.Kinematics({'q2': q2}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()

def get_f_arg_bfp(ana, bfp, q2):
    get_gof(ana, bfp)
    return eos.Observable.make("0->pipi::Arg{f_+}(q2)",
                        ana.parameters,
                        eos.Kinematics({'q2': q2}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()

def get_f_arg_of_psi_bfp(ana, bfp, psi):
    get_gof(ana, bfp)
    return eos.Observable.make("0->pipi::Arg{f_+}(Re{psi},Im{psi})@BHKMNR2026",
                        ana.parameters,
                        eos.Kinematics({'Re{psi}': np.real(psi), 'Im{psi}': np.imag(psi)}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()

def get_f_abs_of_psi_bfp(ana, bfp, psi):
    get_gof(ana, bfp)
    return eos.Observable.make("0->pipi::Abs{f_+}^2(Re{psi},Im{psi})@BHKMNR2026",
                        ana.parameters,
                        eos.Kinematics({'Re{psi}': np.real(psi), 'Im{psi}': np.imag(psi)}),
                        eos.Options({'form-factors': "BHKMNR2026", "n-resonances": '1'})).evaluate()

In [ ]:
def show_rho_mass_width(posterior):
    samples_path = os.path.join(
        BASE_DIRECTORY, "data", posterior, "samples"
    )
    f = eos.data.ImportanceSamples(samples_path)

    names = [
        "0->pipi::M_(+,1,0)@BHKMNR2026",
        "0->pipi::Gamma_(+,1,0)@BHKMNR2026",
    ]

    indices = [f.lookup_table[name] for name in names]
    values = f.samples[:, indices]

    q16 = np.empty(2)
    medians = np.empty(2)
    q84 = np.empty(2)

    for i in range(2):
        q16[i], medians[i], q84[i] = wquantiles(
            values[:, i], f.weights
        )

    error_minus = medians - q16
    error_plus = q84 - medians

    for i, quantity in enumerate(["mass", "width"]):
        print(
            f"rho {quantity} = {medians[i]:.4f} "
            f"−{error_minus[i]:.4f} +{error_plus[i]:.4f} GeV"
        )

    output_path = os.path.join(
        BASE_DIRECTORY, "data", posterior, "rho-mass-width.npz"
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    summary = {
        "parameters": np.array(["mass", "width"]),
        "unit": "GeV",
        "values": values,
        "weights": f.weights,
        "q16": q16,
        "q50": medians,
        "q84": q84,
        "error_minus": error_minus,
        "error_plus": error_plus,
    }

    np.savez_compressed(output_path, **summary)
    print(f"Saved quantities: {output_path}")

    figure_args = {
        "watermark": {"position": "lower right"},
        "plot": {
            "xaxis": {
                "label": r"$M_\rho$",
                "unit": r"$\mathrm{GeV}$",
                "range": [0.755, 0.77],
            },
            "yaxis": {
                "label": r"$\Gamma_\rho$",
                "unit": r"$\mathrm{GeV}$",
                "range": [0.135, 0.15],
            },
            "legend": {"position": "lower right"},
            "items": [
                {
                    "type": "kde2D",
                    "label": "Fit",
                    "color": colors[0],
                    "levels": [68, 95],
                    "contours": ["lines", "labels"],
                    "datafile": samples_path,
                    "variables": names,
                },
                {
                    "type": "errorbars",
                    "positions": [[0.763, 0.145]],
                    "xerrors": [0.002],
                    "yerrors": [0.003],
                    "label": "PDG 2026",
                    "color": "black",
                },
            ],
        },
    }

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()
    plt.show()

    return {**summary, "path": output_path, "figure": figure}

In [ ]:
def show_dispersive_bound(posterior, recompute=False):
    prediction_name = "FF-fp-I1-dispersive-bound"

    prediction_path = os.path.join(
        BASE_DIRECTORY, "data", posterior, f"pred-{prediction_name}"
    )
    summary_path = os.path.join(
        BASE_DIRECTORY, "data", posterior, "dispersive-bound.npz"
    )

    update_prediction = recompute or not os.path.exists(prediction_path)

    if update_prediction:
        eos.tasks.predict_observables(
            af,
            posterior,
            prediction_name,
            base_directory=BASE_DIRECTORY,
        )

    if update_prediction or not os.path.exists(summary_path):
        preds = eos.data.Prediction(prediction_path)
        values = preds.samples[:, 0]
        weights = preds.weights

        q16, q50, q84 = wquantiles(values, weights)

        summary = {
            "values": values,
            "weights": weights,
            "q16": q16,
            "q50": q50,
            "q84": q84,
            "error_minus": q50 - q16,
            "error_plus": q84 - q50,
        }

        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
        np.savez_compressed(summary_path, **summary)

    else:
        with np.load(summary_path) as saved:
            summary = {key: saved[key] for key in saved.files}

    print(
        f"Dispersive bound = {summary['q50']:.4f} "
        f"−{summary['error_minus']:.4f} "
        f"+{summary['error_plus']:.4f}"
    )
    print(f"Saved quantities: {summary_path}")

    plt.figure()
    plt.hist(
        summary["values"],
        weights=summary["weights"],
        bins=np.linspace(0.43, 0.53, 100),
        alpha=0.5,
        label="All samples",
    )
    plt.legend()
    plt.show()

    return {
        **summary,
        "path": summary_path,
        "prediction_path": prediction_path,
    }

In [ ]:
mpi    = eos.Parameters()["mass::pi^+"].evaluate()
sPlus  = 4 * mpi**2
sMinus = 0.0
sIn    = 0.850509 #pi-omega threshold

In [ ]:
def show_scattering_lengths(
    posterior,
    recompute=False,
    xrange=(0.03, 0.085),
    yrange=(-0.02, 0.015),
):
    match = re.fullmatch(r"FF-fp-I1-order(\d+)-(.+)", posterior)
    if match is None:
        raise ValueError(f"Unexpected posterior: {posterior}")

    order, fit_type = match.groups()
    fit_name = {
        "together": "Together",
        "without-Belle": "Without Belle",
    }.get(fit_type, fit_type.replace("-", " "))
    label = f"{fit_name} {order}"

    directory = os.path.join(BASE_DIRECTORY, "data", posterior)
    sl_prediction_path = os.path.join(
        directory, "pred-FF-fp-I1-scattering-length-params"
    )
    sl_summary_path = os.path.join(
        directory, "scattering-lengths", "summary.npz"
    )
    sl_samples_path = os.path.join(
        directory, "scattering-lengths", "samples"
    )

    update_prediction = recompute or not os.path.exists(sl_prediction_path)
    update_summary = update_prediction or not os.path.exists(sl_summary_path)

    if update_prediction:
        eos.tasks.predict_observables(
            af, posterior, "FF-fp-I1-scattering-length-params",
            base_directory=BASE_DIRECTORY,
        )

    if update_summary:
        sl_params = eos.data.Prediction(sl_prediction_path)
        samples = np.asarray(sl_params.samples)

        if samples.ndim != 2 or samples.shape[1] != 4:
            raise ValueError(
                "Expected four prediction columns: r2, r3, r4, r5."
            )

        r2, r3, r4, r5 = samples.T
        weights = np.asarray(sl_params.weights)

        # Calculate a11 and b11 for every sample.
        a11_values = sPlus * (
            -0.36294823212268046 * r2
            +0.06946304407045559 * r3
        ).real

        b11_values = sPlus**2 * (
             0.03311321577857029 * r2
            -0.23706596387010614 * r2**2
            -0.14455309077802547 * r3
            +0.04537099795088154 * r2*r3
            +0.07902198795670203 * r4
            -0.00453709979508815 * r5
        ).real

        q16, medians, q84 = np.column_stack((
            wquantiles(a11_values, weights),
            wquantiles(b11_values, weights),
        ))

        summary = {
            "parameters": np.array(["a11", "b11"]),
            "values": np.column_stack((a11_values, b11_values)),
            "weights": weights,
            "q16": q16,
            "q50": medians,
            "q84": q84,
            "error_minus": medians - q16,
            "error_plus": q84 - medians,
        }

        os.makedirs(os.path.dirname(sl_summary_path), exist_ok=True)
        np.savez_compressed(sl_summary_path, **summary)

    else:
        with np.load(sl_summary_path) as saved:
            summary = {key: saved[key] for key in saved.files}

    # Save the EOS sample format needed by the contour plot.
    if update_summary or not os.path.exists(sl_samples_path):
        parameters = eos.Parameters()
        parameter_list = []

        for name, latex in (
            ("derived::a11", r"a_{11}"),
            ("derived::b11", r"b_{11}"),
        ):
            if not parameters.has(name):
                parameters.declare_and_insert(
                    name, latex, eos.Unit.Unity(),
                    0.0, -100.0, 100.0,
                )
            parameter_list.append(parameters[name])

        os.makedirs(os.path.dirname(sl_samples_path), exist_ok=True)
        eos.data.ImportanceSamples.create(
            sl_samples_path,
            parameter_list,
            summary["values"],
            summary["weights"],
        )

    for i, quantity in enumerate(("a11", "b11")):
        print(
            f"{label} {quantity} = {summary['q50'][i]:.4f} "
            f"(+{summary['error_plus'][i]:.4f}"
            f"/-{summary['error_minus'][i]:.4f})"
        )

    figure_args = {
        "watermark": {"position": "lower left"},
        "plot": {
            "xaxis": {"label": r"$a_1^1$", "range": list(xrange)},
            "yaxis": {"label": r"$b_1^1$", "range": list(yrange)},
            "legend": {"position": "upper right"},
            "items": [
                {
                    "type": "kde2D",
                    "label": label,
                    "color": colors[0],
                    "levels": [68, 95],
                    "contours": ["lines", "labels"],
                    "bandwidth": 1.5,
                    "datafile": sl_samples_path,
                    "xrange": list(xrange),
                    "yrange": list(yrange),
                    "variables": ["derived::a11", "derived::b11"],
                },
                {
                    "type": "errorbars",
                    "label": "CGL 2001",
                    "positions": [[0.0379, 0.00567]],
                    "xerrors": [0.0005],
                    "yerrors": [0.00013],
                    "color": "black",
                },
            ],
        },
    }

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()
    plt.show()

    print(f"Saved quantities: {sl_summary_path}")
    return {
        **summary,
        "path": sl_summary_path,
        "samples_path": sl_samples_path,
        "prediction_path": sl_prediction_path,
        "figure": figure,
    }

In [ ]:
# Sample independent Gaussian literature values (these values are from table 2 of arxiv:hep-ph/013088v1).
rng = np.random.default_rng(42)
n_samples = 2000

a11_theory = rng.normal(0.0379, 0.0005, n_samples)
b11_theory = rng.normal(0.00567, 0.00013, n_samples)

# All sampled curves have equal weight.
theory_weights = np.full(n_samples, 1.0 / n_samples)

# Physical s values, from threshold to 0.2 GeV^2.
s_theory = np.linspace(sPlus, 0.2, 200)
eps = s_theory - sPlus

# Each row is one complete curve from one sampled (a11, b11) pair.
t_theory_samples = eps[None, :] / sPlus * ( a11_theory[:, None] + b11_theory[:, None] * eps[None, :] / sPlus)

# Save the parameter samples, curve samples, grid, and weights.
theory_path = os.path.join(BASE_DIRECTORY, 'data', 'theory', 't11-threshold-samples.npz')
os.makedirs(os.path.dirname(theory_path), exist_ok=True)

np.savez_compressed(theory_path, s=s_theory, a11=a11_theory, b11=b11_theory, t11=t_theory_samples, weights=theory_weights)

In [ ]:
def show_t11_with_theory(posterior, theory_path):
    # The same theory band is used for all truncation orders.
    theory_band_path = (
        os.path.splitext(theory_path)[0] + "-band.npz"
    )

    if not os.path.exists(theory_band_path):
        with np.load(theory_path) as saved:
            s_theory = saved["s"]
            t_theory_samples = saved["t11"]

        # Equal-weight theory samples; preserve your original percentiles.
        t_lower, t_median, t_upper = np.quantile(
            t_theory_samples,
            [0.15865, 0.5, 0.84135],
            axis=0,
        )

        band = {
            "s": s_theory,
            "lower": t_lower,
            "median": t_median,
            "upper": t_upper,
        }

        np.savez_compressed(theory_band_path, **band)

    else:
        with np.load(theory_band_path) as saved:
            band = {key: saved[key] for key in saved.files}

    real_prediction_path = os.path.join(
        BASE_DIRECTORY, "data", posterior, "pred-real-partial-wave-I1"
    )
    imag_prediction_path = os.path.join(
        BASE_DIRECTORY, "data", posterior, "pred-imag-partial-wave-I1"
    )

    figure_args = {
        "watermark": {"position": "lower right"},
        "plot": {
            "xaxis": {
                "label": "$s$",
                "unit": r"$\mathrm{GeV}^2$",
                "range": [0, 0.2],
            },
            "yaxis": {
                "label": r"$t_1^1$",
                "range": [-0.15, 0.15],
            },
            "legend": {"position": "upper left"},
            "items": [
                {
                    "type": "uncertainty",
                    "label": "real part",
                    "interpolation": "cubic",
                    "resolution": 3000,
                    "variable": "q2",
                    "datafile": real_prediction_path,
                },
                {
                    "type": "uncertainty",
                    "label": "imag part",
                    "interpolation": "cubic",
                    "resolution": 3000,
                    "variable": "q2",
                    "datafile": imag_prediction_path,
                },
                {
                    "type": "expression",
                    "expression": "0",
                    "range": [0, 0.2],
                    "color": "k",
                },
            ],
        },
    }

    figure = eos.figure.FigureFactory.from_dict(**figure_args)
    figure.draw()
    ax = figure._ax

    ax.fill_between(
        band["s"],
        band["lower"],
        band["upper"],
        color="red",
        alpha=0.3,
        label="CGL 2001",
    )

    ax.plot(
        band["s"],
        band["median"],
        color="red",
        linewidth=1.5,
    )

    ax.axvline(x=0.0, color="k", linewidth=1.0)
    ax.axvline(
        x=sPlus,
        color="k",
        linestyle=":",
        linewidth=1.0,
        label=r"$s_+$",
    )

    ax.legend(loc="upper left")
    plt.show()

    return {
        **band,
        "path": theory_band_path,
        "figure": figure,
    }

In [ ]:
def charge_radius_squared_calculator(fprime_values):
    # 6 * df/dpsi at psi(0) * dpsi/ds at s=0 * (hbar c)^2
    return 6 * fprime_values * (-1.31241754914090000000) * (0.19732697178003900000)**2 #fm^2

def show_charge_radius(posterior, recompute=False):
    radius_prediction_path = os.path.join(
        BASE_DIRECTORY,
        "data",
        posterior,
        "pred-FF-fp-I1-pion-charge-radius-param",
    )
    radius_summary_path = os.path.join(
        BASE_DIRECTORY,
        "data",
        posterior,
        "charge-radius-squared.npz",
    )

    update_radius_prediction = (
        recompute or not os.path.exists(radius_prediction_path)
    )

    if update_radius_prediction:
        eos.tasks.predict_observables(
            af,
            posterior,
            "FF-fp-I1-pion-charge-radius-param",
            base_directory=BASE_DIRECTORY,
        )

    summary = None

    if (
        not update_radius_prediction
        and os.path.exists(radius_summary_path)
    ):
        with np.load(radius_summary_path) as saved:
            summary = {key: saved[key] for key in saved.files}

    # Also update older files that do not contain asymmetric errors.
    if (
        summary is None
        or "error_minus" not in summary
        or "error_plus" not in summary
    ):
        radius_preds = eos.data.Prediction(radius_prediction_path)

        values = charge_radius_squared_calculator(
            radius_preds.samples[:, 0]
        )
        weights = radius_preds.weights

        q16, q50, q84 = wquantiles(values, weights)

        summary = {
            "values": values,
            "weights": weights,
            "q16": q16,
            "q50": q50,
            "q84": q84,
            "error_minus": q50 - q16,
            "error_plus": q84 - q50,
        }

        os.makedirs(os.path.dirname(radius_summary_path), exist_ok=True)
        np.savez_compressed(radius_summary_path, **summary)

    print(
        f"<r_pi^2> = {summary['q50']:.4f} "
        f"−{summary['error_minus']:.4f} "
        f"+{summary['error_plus']:.4f} fm^2"
    )
    print(f"Saved quantities: {radius_summary_path}")

    return {
        **summary,
        "path": radius_summary_path,
        "prediction_path": radius_prediction_path,
    }

In [ ]:
COUPLING_COLUMNS = [
    ("derived::Re{g_rho_gamma}", r"$\mathrm{Re}\,g_{\rho\gamma}$"),
    ("derived::Im{g_rho_gamma}", r"$\mathrm{Im}\,g_{\rho\gamma}$"),
    ("derived::Re{g_rho_pi_pi}", r"$\mathrm{Re}\,g_{\rho\pi\pi}$"),
    ("derived::Im{g_rho_pi_pi}", r"$\mathrm{Im}\,g_{\rho\pi\pi}$"),
]

coupling_parameters = eos.Parameters()

for name, latex in COUPLING_COLUMNS:
    if not coupling_parameters.has(name):
        coupling_parameters.declare_and_insert(
            name,
            latex,
            eos.Unit.Unity(),
            0.0,
            -100.0,
            100.0,
        )

coupling_parameter_list = [
    coupling_parameters[name] for name, _ in COUPLING_COLUMNS
]


def residue_file_tag(posterior):
    match = re.fullmatch(
        r"FF-fp-I1-order(\d+)-(together|without-Belle)",
        posterior,
    )

    if match is None:
        raise ValueError(f"Unexpected posterior: {posterior}")

    order, fit_type = match.groups()

    if fit_type == "together":
        return f"together{order}"

    return f"wBelle{order}"


def export_rho_residues_psi(posterior, recompute=False):
    """Export residue samples and mass/width for Mathematica."""
    file_tag = residue_file_tag(posterior)
    fit_directory = os.path.join(BASE_DIRECTORY, "data", posterior)

    fit = eos.data.ImportanceSamples(
        os.path.join(fit_directory, "samples")
    )

    residue_parts = []

    for part in ("re", "im"):
        residue_prediction_name = f"{part}-rho-residue-psi"
        residue_prediction_path = os.path.join(
            fit_directory,
            f"pred-{residue_prediction_name}",
        )

        if recompute or not os.path.exists(residue_prediction_path):
            eos.tasks.predict_observables(
                af,
                posterior,
                residue_prediction_name,
                base_directory=BASE_DIRECTORY,
            )

        residue_prediction = eos.data.Prediction(
            residue_prediction_path
        )
        residue_parts.append(residue_prediction.samples[:, 0])

    # Columns: Re(residue_psi), Im(residue_psi), M, Gamma.
    data = np.column_stack((
        residue_parts[0],
        residue_parts[1],
        fit.samples[:, :2],
    ))

    if not np.isfinite(data).all():
        raise ValueError(f"{posterior}: non-finite WL input")

    body = ",\n".join(
        f"{{{r:.17g}, {i:.17g}, {m:.17g}, {g:.17g}}}"
        for r, i, m, g in data
    ).replace("e", "*^")

    wl_path = os.path.join(
        BASE_DIRECTORY,
        f"residues_psi_{file_tag}.wl",
    )

    with open(wl_path, "w", encoding="utf-8") as file:
        file.write("{\n" + body + "\n}")

    print(f"Wrote {wl_path} ({len(data)} rows)")
    return wl_path


def save_rho_couplings(posterior, recompute=False):
    """Compute and save coupling samples, or reuse existing samples."""
    fit_directory = os.path.join(BASE_DIRECTORY, "data", posterior)

    couplings_path = os.path.join(
        fit_directory,
        "rho-couplings",
        "samples",
    )

    if not recompute and os.path.exists(couplings_path):
        print(f"Using saved coupling samples: {couplings_path}")
        return couplings_path

    file_tag = residue_file_tag(posterior)

    fit = eos.data.ImportanceSamples(
        os.path.join(fit_directory, "samples")
    )

    csv_path = os.path.join(
        BASE_DIRECTORY,
        f"residues_s_{file_tag}.csv",
    )

    residue_data = np.loadtxt(
        csv_path,
        delimiter=",",
        ndmin=2,
    )

    if residue_data.shape != (len(fit.samples), 2):
        raise ValueError(
            f"{csv_path}: expected shape {(len(fit.samples), 2)}, "
            f"got {residue_data.shape}"
        )

    res_s = residue_data[:, 0] + 1j * residue_data[:, 1]

    M = fit.samples[:, 0]
    Gamma = fit.samples[:, 1]
    s_rho = (M - 1j * Gamma / 2)**2

    print(f"{posterior}: evaluating couplings...", flush=True)

    analysis = af.analysis(posterior)

    f_components = np.asarray([
        get_f_rho(sample, analysis)
        for sample in fit.samples
    ])
    F_I = f_components[:, 0] + 1j * f_components[:, 1]

    sigma = np.sqrt(1 - sPlus / s_rho)

    g_rho_gamma = np.sqrt(
        24j * np.pi * s_rho / (F_I * res_s * sigma**3)
    )
    g_rho_pi_pi = -(res_s / s_rho) * g_rho_gamma

    values = np.column_stack((
        g_rho_gamma.real,
        g_rho_gamma.imag,
        g_rho_pi_pi.real,
        g_rho_pi_pi.imag,
    ))

    if not np.isfinite(values).all():
        raise ValueError(f"{posterior}: non-finite couplings")

    for parameter, column in zip(
        coupling_parameter_list, values.T
    ):
        parameter.set_min(float(column.min()))
        parameter.set_max(float(column.max()))

    os.makedirs(os.path.dirname(couplings_path), exist_ok=True)

    eos.data.ImportanceSamples.create(
        couplings_path,
        coupling_parameter_list,
        values,
        fit.weights,
    )

    print(f"Saved {len(values)} samples to {couplings_path}")
    return couplings_path


def show_rho_couplings(posterior):
    """Save and print component-wise medians and asymmetric errors."""
    couplings_path = os.path.join(
        BASE_DIRECTORY,
        "data",
        posterior,
        "rho-couplings",
        "samples",
    )

    saved = eos.data.ImportanceSamples(couplings_path)

    values = saved.samples
    weights = saved.weights

    q16 = np.empty(4)
    medians = np.empty(4)
    q84 = np.empty(4)

    for i in range(4):
        q16[i], medians[i], q84[i] = wquantiles(
            values[:, i], weights
        )

    error_minus = medians - q16
    error_plus = q84 - medians

    summary_path = os.path.join(
        BASE_DIRECTORY,
        "data",
        posterior,
        "rho-couplings",
        "summary.npz",
    )

    summary = {
        "parameters": np.array([
            name for name, _ in COUPLING_COLUMNS
        ]),
        "values": values,
        "weights": weights,
        "q16": q16,
        "q50": medians,
        "q84": q84,
        "error_minus": error_minus,
        "error_plus": error_plus,
    }

    np.savez_compressed(summary_path, **summary)

    for quantity, index in (
        ("g_rho_gamma", 0),
        ("g_rho_pi_pi", 2),
    ):
        re = index
        im = index + 1

        print(
            f"{quantity} = "
            f"({medians[re]:.4f} "
            f"−{error_minus[re]:.4f} +{error_plus[re]:.4f}) "
            f"+ i * "
            f"({medians[im]:.4f} "
            f"−{error_minus[im]:.4f} +{error_plus[im]:.4f})"
        )

    print(f"Saved quantities: {summary_path}")

    return {
        **summary,
        "path": summary_path,
        "samples_path": couplings_path,
    }

In [ ]:
def show_a_coefficients(posterior, recompute=False):
    match = re.fullmatch(
        r"FF-fp-I1-order(\d+)-(.+)", posterior
    )
    if match is None:
        raise ValueError(f"Unexpected posterior: {posterior}")

    order = int(match.group(1))
    if order < 4:
        raise ValueError("Expected truncation order >= 4.")

    parameter_names = np.array([
        f"a_{i}" for i in range(order + 1)
    ])

    fit_directory = os.path.join(
        BASE_DIRECTORY, "data", posterior
    )
    summary_path = os.path.join(
        fit_directory, "a-coefficients.npz"
    )

    summary = None

    if not recompute and os.path.exists(summary_path):
        with np.load(summary_path) as saved:
            loaded = {
                key: saved[key] for key in saved.files
            }

        # Reject an old summary that stops before the current order.
        if np.array_equal(
            loaded.get("parameters"), parameter_names
        ):
            summary = loaded
            print(f"Using saved coefficients: {summary_path}")

    if summary is None:
        fit = eos.data.ImportanceSamples(
            os.path.join(fit_directory, "samples")
        )
        weights = np.asarray(fit.weights)

        # Columns 0 and 1: mass and width.
        # Columns 2 through order-2: a4 through a_order.
        fit_coefs = fit.samples[:, 2:order - 1]

        if fit_coefs.shape[1] != order - 3:
            raise ValueError(
                f"Order {order}: expected {order - 3} fit "
                "coefficient columns after mass and width."
            )

        coefficient_columns = []

        for i in range(4):
            coefficient_name = f"a_{i}"
            coefficient_prediction_path = os.path.join(
                fit_directory, f"pred-{coefficient_name}"
            )

            if recompute or not os.path.exists(
                coefficient_prediction_path
            ):
                eos.tasks.predict_observables(
                    af,
                    posterior,
                    coefficient_name,
                    base_directory=BASE_DIRECTORY,
                )

            coefficient_prediction = eos.data.Prediction(
                coefficient_prediction_path
            )

            if coefficient_prediction.samples.shape != (
                len(fit.samples), 1
            ):
                raise ValueError(
                    f"{coefficient_name}: expected one coefficient "
                    "value per fit sample."
                )

            prediction_weights = np.asarray(
                coefficient_prediction.weights
            )

            if (
                prediction_weights.shape != weights.shape
                or not np.allclose(
                    prediction_weights / prediction_weights.sum(),
                    weights / weights.sum(),
                    rtol=1e-6,
                    atol=1e-12,
                )
            ):
                raise ValueError(
                    f"{coefficient_name}: weights do not match the fit."
                )

            coefficient_columns.append(
                coefficient_prediction.samples[:, 0]
            )

        all_coefs = np.column_stack(
            coefficient_columns + [fit_coefs]
        )

        q16, medians, q84 = np.array([
            [
                quantile_1D(all_coefs[:, i], weights, q)
                for i in range(all_coefs.shape[1])
            ]
            for q in (0.16, 0.50, 0.84)
        ])

        summary = {
            "parameters": parameter_names,
            "values": all_coefs,
            "weights": weights,
            "q16": q16,
            "q50": medians,
            "q84": q84,
            "error_minus": medians - q16,
            "error_plus": q84 - medians,
        }

        os.makedirs(fit_directory, exist_ok=True)
        np.savez_compressed(summary_path, **summary)
        print(f"Saved coefficients: {summary_path}")

    for name, median, err_low, err_high in zip(
        summary["parameters"],
        summary["q50"],
        summary["error_minus"],
        summary["error_plus"],
    ):
        print(
            f"{name} = {median:.4f} "
            f"(+{err_high:.4f}/-{err_low:.4f})"
        )

    n_coefs = len(summary["q50"])
    xs = np.arange(n_coefs)

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.errorbar(
        xs,
        summary["q50"],
        yerr=[
            summary["error_minus"],
            summary["error_plus"],
        ],
        fmt="o",
        capsize=5,
        capthick=1.5,
        elinewidth=1.5,
        markersize=7,
        color="b",
        ecolor="b",
    )

    ax.axhline(
        0,
        color="gray",
        linewidth=0.8,
        linestyle="--",
        alpha=0.5,
    )
    ax.set_xticks(xs)
    ax.set_xticklabels([
        rf"$a_{{{i}}}$" for i in range(n_coefs)
    ])
    ax.set_ylabel("value")

    fig.tight_layout()
    plt.show()

    return {
        **summary,
        "path": summary_path,
        "figure": fig,
    }

In [ ]:
def show_results_table(posterior, recompute_gof=False):
    match = re.fullmatch(
        r"FF-fp-I1-order(\d+)-(.+)", posterior
    )
    if match is None:
        raise ValueError(f"Unexpected posterior name: {posterior}")

    order, fit_type = match.groups()

    fit_name = {
        "together": "Together",
        "without-Belle": "Without Belle",
    }.get(fit_type, fit_type.replace("-", " "))

    label = f"{fit_name} order {order}"
    fit_directory = os.path.join(BASE_DIRECTORY, "data", posterior)

    def load_summary(filename):
        path = os.path.join(fit_directory, filename)
        with np.load(path) as saved:
            return {key: saved[key] for key in saved.files}

    # Load the previously saved numerical summaries.
    mass_width = load_summary("rho-mass-width.npz")
    dispersive_bound = load_summary("dispersive-bound.npz")
    scattering_lengths = load_summary(
        os.path.join("scattering-lengths", "summary.npz")
    )
    radius = load_summary("charge-radius-squared.npz")
    couplings = load_summary(
        os.path.join("rho-couplings", "summary.npz")
    )

    def summary(saved, column=0, digits=4, factor=1.0):
        # Handles both scalar summaries and multi-column summaries.
        lower, median, upper = [
            factor * float(
                np.asarray(saved[key]).reshape(-1)[column]
            )
            for key in ("q16", "q50", "q84")
        ]

        return (
            f"{median:.{digits}f} "
            f"(+{upper - median:.{digits}f}"
            f"/-{median - lower:.{digits}f})"
        )

    gof_path = os.path.join(
        fit_directory,
        "goodness-of-fit-without-root.npz",
    )

    if recompute_gof or not os.path.exists(gof_path):
        # Evaluate goodness of fit at the saved mode.
        full_analysis = af.analysis(posterior)
        mode = eos.data.Mode(
            os.path.join(fit_directory, "mode-default")
        )

        best_fit_values = {
            parameter.name(): value
            for parameter, value in zip(
                full_analysis.varied_parameters, mode.mode
            )
        }

        for parameter in full_analysis.varied_parameters:
            full_analysis.parameters.set(
                parameter.name(),
                best_fit_values[parameter.name()],
            )

        full_gof = full_analysis.goodness_of_fit()

        # Remove the root-penalty constraint.
        reduced_args = dict(full_analysis.init_args)
        manual_constraints = dict(
            reduced_args.get("manual_constraints") or {}
        )

        root_names = [
            name for name in manual_constraints
            if str(name) == "0->pipi::root_penalty"
        ]

        if len(root_names) != 1:
            raise ValueError(
                "Expected one root-penalty constraint; "
                f"found {root_names}."
            )

        del manual_constraints[root_names[0]]
        reduced_args["manual_constraints"] = manual_constraints

        reduced_analysis = eos.Analysis(**reduced_args)

        for parameter in reduced_analysis.varied_parameters:
            reduced_analysis.parameters.set(
                parameter.name(),
                best_fit_values[parameter.name()],
            )

        chi2_without_root = (
            reduced_analysis.goodness_of_fit().total_chi_square()
        )

        # Preserve your existing degrees-of-freedom convention.
        dof_without_root = (
            full_gof.total_degrees_of_freedom() - 1
        )

        if dof_without_root <= 0:
            raise ValueError("Degrees of freedom must be positive.")

        pvalue = 100 * chi2.sf(
            chi2_without_root, dof_without_root
        )

        gof_summary = {
            "chi2": chi2_without_root,
            "dof": dof_without_root,
            "pvalue_percent": pvalue,
        }

        np.savez_compressed(gof_path, **gof_summary)

    else:
        gof_summary = load_summary(
            "goodness-of-fit-without-root.npz"
        )

    pvalue = float(gof_summary["pvalue_percent"])

    row = {
        "Fit": label,
        "p-value (%)": f"{pvalue:.2f}",
        "M_rho (MeV)": summary(
            mass_width, column=0, digits=2, factor=1000
        ),
        "Gamma_rho (MeV)": summary(
            mass_width, column=1, digits=2, factor=1000
        ),
        "Dispersive bound": summary(dispersive_bound),
        "a11": summary(scattering_lengths, column=0),
        "b11": summary(scattering_lengths, column=1),
        "r_pi^2 (fm^2)": summary(radius),
    }

    coupling_indices = {
        str(name): i
        for i, name in enumerate(couplings["parameters"])
    }

    coupling_labels = (
        "Re(g_rhogamma)",
        "Im(g_rhogamma)",
        "Re(g_rhopipi)",
        "Im(g_rhopipi)",
    )

    for column_label, (name, _) in zip(
        coupling_labels, COUPLING_COLUMNS
    ):
        row[column_label] = summary(
            couplings,
            column=coupling_indices[name],
            digits=3,
        )

    table = pd.DataFrame([row]).set_index("Fit")
    table.attrs["gof_path"] = gof_path

    return table